# CodeBERT experiments on Kaggle

Before running, in the right-hand **Session options** panel:
- **Accelerator:** GPU T4 x2 (or P100)
- **Internet:** On (needed to clone the repo and download `microsoft/codebert-base`)

Then **Save Version -> Save & Run All (Commit)**. When it finishes, download `/kaggle/working/` from the **Output** tab
(`results.zip` has everything). Unzip it into `experiments/` in the repo.

In [ ]:
REPO_URL = "https://github.com/alexsmyr0/thesis-experiments"
# Private repo? Add a GitHub token under Add-ons -> Secrets (name it GITHUB_TOKEN), then uncomment:
# from kaggle_secrets import UserSecretsClient
# REPO_URL = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")

!rm -rf /tmp/thesis-experiments
!git clone --depth 1 {REPO_URL} /tmp/thesis-experiments
# transformers>=5 is needed (warmup_steps=0.1 as a ratio); Kaggle preinstalls an older version.
!pip install -q -U "transformers>=5" peft datasets accelerate

In [ ]:
import torch, transformers, peft, sklearn
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__, "| sklearn", sklearn.__version__)
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU in Session options, then restart"

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

REPO = Path("/tmp/thesis-experiments")
OUT = Path("/kaggle/working/experiments")

# Datasets in data/v2/ to run. The first two were already run on the M2 Air; rerunning them here gives a
# same-hardware control for comparing against the other three.
DATASETS = ["control_unmodified", "only_modules", "only_comments", "only_newlines", "neutral_all"]
CODEBERT = "both"      # "full", "lora" or "both"
FP16 = False           # True is faster on a T4 but can shift scores slightly; keep False for thesis settings

for name in DATASETS:
    print(f"\n######## {name} ########", flush=True)
    env = {**os.environ,
           "THESIS_CSV": str(REPO / "data" / "v2" / f"{name}.csv"),
           "THESIS_OUT": str(OUT / name),
           "PY": sys.executable,
           "CODEBERT_ARGS": "--fp16" if FP16 else ""}
    subprocess.run(["sh", str(REPO / "scripts" / "run_all.sh"), f"--codebert={CODEBERT}"], env=env, check=True)
    shutil.rmtree(OUT / name / "checkpoints", ignore_errors=True)  # keep the Output small

In [ ]:
for name in DATASETS:
    print(f"\n=== {name} ===")
    print((OUT / name / "logs" / "summary.txt").read_text())
shutil.make_archive("/kaggle/working/results", "zip", OUT)
print("Wrote /kaggle/working/results.zip")